<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/encode_categories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

In [ ]:
%%writefile /content/drive/MyDrive/ml_project/encode_categories.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

target_dir = "/content/drive/MyDrive/ml_project"
input_pickle = os.path.join(target_dir, "df_lags.pkl")
choice_file = os.path.join(target_dir, "encode_method.txt")
output_pickle = os.path.join(target_dir, "df_encoded.pkl")

# ---------------------------------------------------------
# encode_categories_ui
# ---------------------------------------------------------
def encode_categories_ui():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_lags.pkl not found")

    df = pd.read_pickle(input_pickle)

    # tylko kolumny typu object / string
    text_cols = [c for c in df.columns if df[c].dtype == object]

    label_cols = widgets.Label("wybierz kolumny do kodowania (wartości tekstowe):")
    select = widgets.SelectMultiple(options=text_cols)

    label_method = widgets.Label("wybierz metodę kodowania (onehot / mean):")
    dropdown_method = widgets.Dropdown(options=["onehot", "mean"])

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        chosen_cols = list(select.value)
        chosen_method = dropdown_method.value

        with open(choice_file, "w") as f:
            f.write(",".join(chosen_cols) + "\n")
            f.write(chosen_method + "\n")

        with out:
            print("saved:", choice_file)
            print("columns:", chosen_cols)
            print("method:", chosen_method)

    btn.on_click(on_click)

    display(widgets.VBox([
        label_cols,
        select,
        label_method,
        dropdown_method,
        btn,
        out
    ]))

# ---------------------------------------------------------
# encode_categories
# ---------------------------------------------------------
def encode_categories():

    if not os.path.exists(input_pickle):
        raise FileNotFoundError("df_lags.pkl not found")

    if not os.path.exists(choice_file):
        raise FileNotFoundError("encode_method.txt not found")

    df = pd.read_pickle(input_pickle)

    with open(choice_file, "r") as f:
        lines = f.read().strip().split("\n")

    cols_raw = lines[0]
    method = lines[1]

    cols = [c.strip() for c in cols_raw.split(",") if c.strip()]

    # one-hot encoding
    if method == "onehot":
        df = pd.get_dummies(df, columns=cols, dtype=int)

    # mean encoding
    elif method == "mean":
        # wymaga kolumny target — w twoim projekcie jest zwykle 'y' albo 'target'
        # przyjmuję nazwę 'target' (możesz zmienić)
        if "target" not in df.columns:
            raise ValueError("mean encoding wymaga kolumny 'target' w dataframe")

        for col in cols:
            means = df.groupby(col)["target"].mean()
            df[col] = df[col].map(means)

    df.to_pickle(output_pickle)

    print("saved:", output_pickle)
    print("encoded categories:", cols)
    print("method used:", method)

    return df
